In [ ]:
from abc import ABC, abstractmethod
import pandas as pd
import numpy as np
import yfinance as yf
import plotly.graph_objects as go  

In [ ]:
# ==========================================
# 1. CLASSE GENITORE (BASE)
# ==========================================
class BaseCryptoStrategy(ABC):
    """
    Classe genitore che gestisce il recupero dati, le statistiche e i grafici interattivi.
    Le classi figlie devono implementare solo la logica di 'run_strategy'.
    """
    
    def __init__(self, initial_value: float = 10000.0):
        self.initial_value = initial_value
        self.data_dict = {}
        self.assets = []
        self.index_df = None
        self.benchmark_df = None  
        self.benchmark_ticker = 'BTC-USD' 
        self.rebalance_dates = [] # <-- Nuovo attributo per tracciare le date di ribilanciamento

    def import_data(self, assets: list, timeframe: str = "1y"):
        """Scarica e prepara i dati base (Prezzo, Volume, Circulating Supply) per la lista di asset."""
        print(f"Inizio download dati per il periodo '{timeframe}'...")
        self.assets = []
        
        for ticker_symbol in assets:
            try:
                ticker = yf.Ticker(ticker_symbol)
                df_raw = ticker.history(period=timeframe)
                
                if df_raw.empty:
                    print(f"Nessun dato per {ticker_symbol}. Salto.")
                    continue
                    
                circulating_supply = ticker.info.get('circulatingSupply')
                if not circulating_supply:
                    print(f"Circulating Supply mancante per {ticker_symbol}. Salto.")
                    continue
                    
                df_final = df_raw[['Close', 'Volume']].copy()
                df_final['Circulating Supply'] = circulating_supply
                df_final.columns = ['Prezzo di Chiusura', 'Volume 24h', 'Circulating Supply']
                
                self.data_dict[ticker_symbol] = df_final
                self.assets.append(ticker_symbol)
                
            except Exception as e:
                print(f"Errore durante il recupero di {ticker_symbol}: {e}")

        # --- GESTIONE DEL BENCHMARK ---
        if self.benchmark_ticker in self.data_dict:
            self.benchmark_df = self.data_dict[self.benchmark_ticker]
        else:
            print(f"\n{self.benchmark_ticker} non presente nel dizionario. Avvio download in un df separato...")
            try:
                ticker_bench = yf.Ticker(self.benchmark_ticker)
                df_bench_raw = ticker_bench.history(period=timeframe)
                
                if not df_bench_raw.empty:
                    df_bench_final = df_bench_raw[['Close']].copy()
                    df_bench_final.columns = ['Prezzo di Chiusura']
                    self.benchmark_df = df_bench_final
                    print(f"Benchmark {self.benchmark_ticker} completato con successo.\n")
                else:
                    print(f"Nessun dato trovato per il benchmark {self.benchmark_ticker}.\n")
            except Exception as e:
                print(f"Errore durante il download del benchmark {self.benchmark_ticker}: {e}\n")

    @abstractmethod
    def run_strategy(self, **kwargs):
        """Metodo astratto: ogni classe figlia deve definire la propria logica di backtest."""
        pass

    def plot_results(self):
        """Grafica l'andamento della strategia confrontato con il benchmark (Interattivo)."""
        if self.index_df is None:
            raise ValueError("Strategia non calcolata. Esegui run_strategy() prima di plottare.")
            
        fig = go.Figure()

        # --- PLOT STRATEGIA ---
        fig.add_trace(go.Scatter(
            x=self.index_df.index, 
            y=self.index_df['Valore Indice'],
            mode='lines',
            name='Indice Strategia',
            line=dict(color='#1f77b4', width=2)
        ))
        
        # --- PLOT BENCHMARK ---
        if self.benchmark_df is not None:
            common_idx = self.index_df.index.intersection(self.benchmark_df.index)
            if not common_idx.empty:
                btc_prices = self.benchmark_df['Prezzo di Chiusura'].loc[common_idx]
                btc_normalized = (btc_prices / btc_prices.iloc[0]) * self.initial_value
                
                fig.add_trace(go.Scatter(
                    x=btc_normalized.index, 
                    y=btc_normalized,
                    mode='lines',
                    name=f'{self.benchmark_ticker} (Benchmark)',
                    line=dict(color='#cc8306', width=2)
                ))
        
        # --- LINEA CAPITALE INIZIALE ---
        fig.add_hline(
            y=self.initial_value, 
            line_dash="dash", 
            line_color="red", 
            annotation_text=f"Capitale Iniziale ({self.initial_value}$)", 
            annotation_position="bottom right"
        )

        # --- LINEE DEI RIBILANCIAMENTI ---
        # Aggiungiamo una linea verticale per ogni data in cui è avvenuto un ribilanciamento
        if self.rebalance_dates:
            for date in self.rebalance_dates:
                fig.add_vline(
                    x=date, 
                    line_width=1, 
                    line_dash="dot", 
                    line_color="green",
                    opacity=0.6
                )
            # Aggiungo una riga invisibile solo per far comparire la legenda dei ribilanciamenti
            fig.add_trace(go.Scatter(
                x=[None], y=[None], mode='lines',
                line=dict(color='green', dash='dot'),
                name='Ribilanciamento'
            ))

        # --- LAYOUT INTERATTIVO ---
        fig.update_layout(
            title="Performance Strategia vs BTC-USD",
            xaxis_title="Data",
            yaxis_title="Valore del Portafoglio ($)",
            hovermode="x unified", # Mostra i dati di tutte le curve passando il mouse su un punto
            template="plotly_white",
            legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
        )
        
        fig.show()

    def calculate_stats(self):
        """Calcola e restituisce le statistiche della strategia rispetto al benchmark."""
        if self.index_df is None:
            raise ValueError("Strategia non calcolata. Esegui run_strategy().")
        
        strat_returns = self.index_df['Valore Indice'].pct_change().dropna()
        stats = {'Strategia': self._compute_metrics(strat_returns)}
        
        if self.benchmark_df is not None:
            common_idx = self.index_df.index.intersection(self.benchmark_df.index)
            if not common_idx.empty:
                btc_prices = self.benchmark_df['Prezzo di Chiusura'].loc[common_idx]
                btc_returns = btc_prices.pct_change().dropna()
                stats[self.benchmark_ticker] = self._compute_metrics(btc_returns)
            
        return pd.DataFrame(stats).round(4)

    def _compute_metrics(self, returns: pd.Series) -> dict:
        """Helper interno per il calcolo delle metriche finanziarie su 365 giorni."""
        trading_days = 365
        ann_return = returns.mean() * trading_days
        ann_volatility = returns.std() * np.sqrt(trading_days)
        sharpe_ratio = ann_return / ann_volatility if ann_volatility != 0 else 0
        
        cumulative_returns = (1 + returns).cumprod()
        peak = cumulative_returns.cummax()
        max_drawdown = ((cumulative_returns - peak) / peak).min()
        
        return {
            'Ritorno Annualizzato (%)': ann_return * 100,
            'Volatilità Annualizzata (%)': ann_volatility * 100,
            'Sharpe Ratio': sharpe_ratio,
            'Max Drawdown (%)': max_drawdown * 100
        }

In [2]:
# ==========================================
# 2. CLASSE FIGLIA (STRATEGIA SPECIFICA)
# ==========================================
class VolMktCapStrategy(BaseCryptoStrategy):
    """
    Strategia figlia che pondera il portafoglio in base al rapporto Volume/MarketCap.
    Eredita import_data, plot_results e calculate_stats dal genitore.
    """
    
    def run_strategy(self, rebalance_freq_days: int = 90):
        if not self.data_dict:
            raise ValueError("Nessun dato disponibile. Esegui import_data() per primo.")
            
        print(f"\nAvvio backtest con ribilanciamento ogni {rebalance_freq_days} giorni...")
        
        df_prices = pd.DataFrame()
        df_vol_mktcap = pd.DataFrame()
        
        # Allineamento dati e calcolo metrica specifica
        for ticker in self.assets:
            df = self.data_dict[ticker].copy()
            df['vol/mktcap'] = df['Volume 24h'] / (df['Circulating Supply'] * df['Prezzo di Chiusura'])
            df.index = pd.to_datetime(df.index)
            
            df_prices[ticker] = df['Prezzo di Chiusura']
            df_vol_mktcap[ticker] = df['vol/mktcap']
            
        df_prices.ffill(inplace=True)
        df_vol_mktcap.ffill(inplace=True)
        df_prices.dropna(inplace=True)
        df_vol_mktcap.dropna(inplace=True)
        
        dates = df_prices.index
        
        # --- GESTIONE WARM-UP PER I PESI INIZIALI ---
        if len(dates) <= rebalance_freq_days:
            raise ValueError(f"Dati storici insufficienti. Servono più di {rebalance_freq_days} giorni per calcolare l'allocazione iniziale.")
            
        # 1. Usiamo i primi 'rebalance_freq_days' per calcolare la MEDIA INIZIALE
        warmup_data = df_vol_mktcap.iloc[:rebalance_freq_days]

        # --- NUOVA STAMPA DEL PERIODO DI WARM-UP ---
        warmup_start = warmup_data.index[0].strftime('%Y-%m-%d')
        warmup_end = warmup_data.index[-1].strftime('%Y-%m-%d')
        print(f"Periodo di warm-up ({rebalance_freq_days} giorni): dal {warmup_start} al {warmup_end}")
        # -------------------------------------------

        initial_avg = warmup_data.mean()
        total_initial_avg = initial_avg.sum()
        
        if total_initial_avg > 0:
            weights = (initial_avg / total_initial_avg).to_dict()
        else:
            weights = {ticker: 1.0 / len(self.assets) for ticker in self.assets}
            
        # 2. Il backtest operativo parte DOPO il periodo di warm-up
        backtest_dates = dates[rebalance_freq_days:]
        first_trading_date = backtest_dates[0]
        portfolio_value = self.initial_value
        shares = {}
        
        # Inizializzazione delle shares con i pesi basati sulla media iniziale
        for ticker in self.assets:
            shares[ticker] = (portfolio_value * weights[ticker]) / df_prices.loc[first_trading_date, ticker]
            
        index_values = []
        last_rebalance_date = first_trading_date
        
        # --- LOOP GIORNALIERO SUI GIORNI EFFETTIVI ---
        for current_date in backtest_dates:
            current_portfolio_value = sum(shares[ticker] * df_prices.loc[current_date, ticker] for ticker in self.assets)
            index_values.append(current_portfolio_value)
            
            # Controllo ribilanciamento
            if (current_date - last_rebalance_date).days >= rebalance_freq_days:
                past_data = df_vol_mktcap.loc[last_rebalance_date:current_date]
                avg_vol_mktcap = past_data.mean()
                total_avg = avg_vol_mktcap.sum()
                
                if total_avg > 0:
                    weights = (avg_vol_mktcap / total_avg).to_dict()
                    
                # Ribilanciamento: aggiornamento shares
                for ticker in self.assets:
                    shares[ticker] = (current_portfolio_value * weights[ticker]) / df_prices.loc[current_date, ticker]
                    
                last_rebalance_date = current_date
                
        self.index_df = pd.DataFrame({'Date': backtest_dates, 'Valore Indice': index_values})
        self.index_df.set_index('Date', inplace=True)
        print("Backtest completato.")

In [3]:
# ==========================================
# 3. CLASSE FIGLIA (REBALANCED MARKET CAP STRATEGY - TOP 10 DINAMICA)
# ==========================================
class RebalancedMarketCapStrategy(BaseCryptoStrategy):
    """
    Strategia figlia che seleziona dinamicamente le 10 crypto più capitalizzate.
    Effettua un ribilanciamento periodico (es. mensile): rivaluta l'intero universo
    di asset, estrae le nuove Top 10, ricalcola i pesi in base alla Mkt Cap 
    e rialloca il capitale accumulato fino a quel momento.
    """
    
    def run_strategy(self, rebalance_freq='Q', **kwargs):
        """
        rebalance_freq: Frequenza di ribilanciamento. 
        'M' (Month End) = fine mese, 'Q' (Quarter End) = fine trimestre, 'Y' = fine anno.
        """
        if not self.data_dict:
            raise ValueError("Nessun dato disponibile. Esegui import_data() per primo.")
            
        print(f"\nAvvio backtest: Top 10 Dinamica con ribilanciamento (frequenza: {rebalance_freq})...")
        
        df_prices = pd.DataFrame()
        df_mktcap = pd.DataFrame()
        
        # 1. Allineamento dati e calcolo della Market Cap
        for ticker in self.assets:
            df = self.data_dict[ticker].copy()
            df['MarketCap'] = df['Prezzo di Chiusura'] * df['Circulating Supply']
            
            df.index = pd.to_datetime(df.index)
            df_prices[ticker] = df['Prezzo di Chiusura']
            df_mktcap[ticker] = df['MarketCap']
            
        df_prices.ffill(inplace=True)
        df_mktcap.ffill(inplace=True)
        df_prices.dropna(how='all', inplace=True)
        
        dates = df_prices.index
        first_date = dates[0]
        
        # 2. Identificazione dei giorni di ribilanciamento
        # Usiamo resample per trovare l'ultimo giorno disponibile di ogni mese/trimestre
        rebalance_dates = df_prices.resample(rebalance_freq).last().index
        # Ci assicuriamo che le date calcolate siano effettivamente nell'indice
        rebalance_dates = [d for d in rebalance_dates if d in dates]
        
        portfolio_value = self.initial_value
        shares = {}
        current_top_10 = []
        index_values = []
        
        # 3. Loop giornaliero: simulazione day-by-day
        for current_date in dates:
            
            # --- A. AGGIORNAMENTO VALORE PORTAFOGLIO ---
            if not shares:
                # Giorno 1: il valore è il capitale iniziale
                pass 
            else:
                # Calcola il valore del ptf in base alle quote possedute e ai prezzi di oggi
                portfolio_value = sum(shares[ticker] * df_prices.loc[current_date, ticker] for ticker in current_top_10)
            
            index_values.append(portfolio_value)
            
            # --- B. CONTROLLO RIBILANCIAMENTO ---
            # Ribilanciamo se è il primo giorno assoluto o se è una data di ribilanciamento
            if current_date == first_date or current_date in rebalance_dates:
                
                # Prende la Mkt Cap di TUTTE le crypto in questa specifica data
                daily_mktcap = df_mktcap.loc[current_date].dropna()
                
                # Seleziona le NUOVE Top 10
                top_10 = daily_mktcap.nlargest(10)
                current_top_10 = top_10.index.tolist()
                
                total_mktcap = top_10.sum()
                if total_mktcap == 0:
                    continue # Evita divisioni per zero se i dati sono corrotti
                
                # Ricalcola i nuovi pesi
                weights = (top_10 / total_mktcap).to_dict()
                
                # Vende tutto e ricompra: ricalcola le quote esatte per le nuove Top 10
                shares = {}
                for ticker in current_top_10:
                    allocated_capital = portfolio_value * weights[ticker]
                    price = df_prices.loc[current_date, ticker]
                    # Assegna le nuove quote (se il prezzo è > 0 per evitare errori)
                    shares[ticker] = allocated_capital / price if price > 0 else 0
                    
        # 4. Salvataggio della serie storica
        self.index_df = pd.DataFrame({'Date': dates, 'Valore Indice': index_values})
        self.index_df.set_index('Date', inplace=True)
        
        print(f"Backtest completato. Composizione FINALE del portafoglio al {dates[-1].strftime('%Y-%m-%d')}:")
        for ticker in current_top_10:
            print(f"- {ticker}: {weights[ticker]:.2%} (Mkt Cap: {top_10[ticker]:,.0f})")

In [4]:
if __name__ == "__main__":
    # 1. Definiamo la lista degli asset da includere nel portafoglio
    # È consigliato includere "BTC-USD" affinché venga usato come benchmark nei grafici e nelle statistiche

    Conio_coin = [
        "BTC-USD", "ETH-USD", "XRP-USD", "SOL-USD", "DOGE-USD", 
        "ADA-USD", "LINK-USD", "AVAX-USD", "DOT-USD", 
        "UNI7083-USD", "SKY33038-USD", "NEAR-USD", "ATOM-USD", 
        "POL28321-USD", "ALGO-USD", "APT21794-USD", "ARB11841-USD", "STX4847-USD", 
        "INJ-USD", "TIA-USD", "GRT6719-USD", "OP-USD", "SUI20947-USD", 
        "XLM-USD", "XPL-USD", "ONDO-USD", "HBAR-USD", 
        "FIL-USD", "AAVE-USD", "NIGHT39064-USD"
    ]
    
    # 2. Inizializziamo la strategia figlia con un capitale di partenza
    strategy = RebalancedMarketCapStrategy(initial_value=10000.0)
    
    # 3. Importiamo i dati (es. ultimo anno)
    strategy.import_data(assets=Conio_coin, timeframe="18mo")
    
    # 4. Eseguiamo il backtest (es. ribilanciamento ogni 30 giorni)
    # Avvolgiamo in un try-except in caso di errori di connessione o dati mancanti
    try:
        strategy.run_strategy(rebalance_freq_days=30)
        
        # 5. Calcoliamo e stampiamo le statistiche
        print("\n--- Statistiche di Performance ---")
        stats_df = strategy.calculate_stats()
        print(stats_df)
        
        # 6. Mostriamo il grafico
        strategy.plot_results()
        
    except ValueError as e:
        print(f"\nErrore durante l'esecuzione della strategia: {e}")

Inizio download dati per il periodo '18mo'...

Avvio backtest: Top 10 Dinamica con ribilanciamento (frequenza: Q)...
Backtest completato. Composizione FINALE del portafoglio al 2026-02-24:
- BTC-USD: 74.50% (Mkt Cap: 1,749,658,335,220)
- ETH-USD: 15.25% (Mkt Cap: 358,098,722,397)
- XRP-USD: 4.78% (Mkt Cap: 112,272,543,636)
- SOL-USD: 3.01% (Mkt Cap: 70,790,217,104)
- DOGE-USD: 0.84% (Mkt Cap: 19,807,603,129)
- ADA-USD: 0.51% (Mkt Cap: 12,006,554,963)
- LINK-USD: 0.37% (Mkt Cap: 8,630,328,293)
- XLM-USD: 0.28% (Mkt Cap: 6,593,346,459)
- SUI20947-USD: 0.23% (Mkt Cap: 5,394,645,291)
- AVAX-USD: 0.23% (Mkt Cap: 5,312,904,374)

--- Statistiche di Performance ---
                             Strategia  BTC-USD
Ritorno Annualizzato (%)        6.8937   9.6210
Volatilità Annualizzata (%)    51.4684  46.1179
Sharpe Ratio                    0.1339   0.2086
Max Drawdown (%)              -53.4499 -49.7388
